# ChaseEscapeEnv PPO Training (Colab)

This notebook trains a PPO agent for the 2D pursuit-evasion environment **only in Colab**.

- Uses GPU (e.g. T4) if enabled in Colab
- Keeps your local machine for running the Pygame sandbox only
- Saves the trained model to `runs/ppo_chase_escape/ppo_chase_escape_final.zip`

You can later download that model and play it locally or in Colab.

In [2]:
# If needed, clone your repository here.
# If you're using Cursor's Colab integration and the project is already present,
# just make sure this path matches your project root.

%cd /content

# Example (adjust to your GitHub URL):
# !git clone https://github.com/<your-user>/<your-repo>.git
# %cd /content/<your-repo>/pursuit_arena

# If the repo is already in /content/pursuit_arena, just do:
%cd /content/pursuit_arena

/content
[Errno 2] No such file or directory: '/content/pursuit_arena'
/content


In [ ]:
# Install dependencies in Colab only (NOT on your local machine).

!pip install pygame gymnasium stable-baselines3[extra] numpy

In [ ]:
# Optional: quick environment check
from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeEnv
from stable_baselines3.common.env_checker import check_env

env = ChaseEscapeEnv(render_mode=None)
check_env(env, warn=True)
env.close()

In [ ]:
# Train PPO on ChaseEscapeEnv
from pathlib import Path

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeEnv

log_dir = Path("runs/ppo_chase_escape")
log_dir.mkdir(parents=True, exist_ok=True)

def make_env():
    env = ChaseEscapeEnv(render_mode=None)
    env = Monitor(env, log_dir / "monitor.csv")
    return env

vec_env = DummyVecEnv([make_env])

model = PPO(
    "MlpPolicy",
    vec_env,
    verbose=1,
    tensorboard_log=str(log_dir / "tb"),
)

total_timesteps = 500_000  # adjust as you like
model.learn(total_timesteps=total_timesteps)

model_path = log_dir / "ppo_chase_escape_final.zip"
model.save(str(model_path))
print("Saved model to", model_path)
vec_env.close()

In [ ]:
# Evaluate the trained model in Colab (no rendering window)
from stable_baselines3 import PPO
from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeEnv

env = ChaseEscapeEnv(render_mode=None)
model = PPO.load("runs/ppo_chase_escape/ppo_chase_escape_final.zip", env=env)

episodes = 5
for ep in range(episodes):
    obs, info = env.reset()
    done = False
    truncated = False
    ep_reward = 0.0
    while not (done or truncated):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(action)
        ep_reward += float(reward)
    print(f"Episode {ep+1}: reward={ep_reward:.2f}, info={info}")

env.close()

In [ ]:
# Download the trained model to your local machine (from Colab)
from google.colab import files

files.download("runs/ppo_chase_escape/ppo_chase_escape_final.zip")